# 文档切分器 Text Splitters
## 为什么分割/切分/分块？
获取 Document 对象后，需要将其切分成一个个小块（Chunk）。原因：
- 长文档问题：大模型存在最大输入的 `Token 限制` ，如果一个 `Document 非常大` ，在输入大模型时会 `被截断` ，导致信息缺失。
- 检索精度：Document 可能包含 `非常多无关的信息` ，这些无效信息会 `干扰大模型` 的生成，而小块检索更精准。
- 成本控制：减少不必要的`token消耗`

无论是在存储还是检索过程中，都以这些 `块(chunk)` 为基本单位，这样能有效地避免内容噪声干扰和超出最大 Token 的问题。

## Chunking拆分的策略
方法1：根据句子切分
方法2：按照固定字符数来切分
方法3：按固定字符数来切分，结合重叠窗口
方法4：递归字符切分方法
方法5：根据语义内容切分

In [ ]:
## TextSplitter 源码分析

class TextSplitter(BaseDocumentTransformer, ABC):
    """用于将文本切分为多个块的接口。"""

    def __init__(
        self,
        chunk_size: int = 4000,
        chunk_overlap: int = 200,
        length_function: Callable[[str], int] = len,
        keep_separator: bool | Literal["start", "end"] = False,  # noqa: FBT001,FBT002
        add_start_index: bool = False,  # noqa: FBT001,FBT002
        strip_whitespace: bool = True,  # noqa: FBT001,FBT002
    ) -> None:
        """创建一个新的 `TextSplitter`。

        Args:
            chunk_size: 返回的文本块的最大大小。
            chunk_overlap: 文本块之间重叠的字符数。
            length_function: 用于衡量给定文本块长度的函数。
            keep_separator: 是否保留分隔符，以及将其放在对应文本块中的哪个位置
                `(True='start')`。
            add_start_index: 如果为 `True`，则在元数据中包含文本块的起始索引。
            strip_whitespace: 如果为 `True`，则去除每个文档开头和结尾的空白字符。

        Raises:
            ValueError: 如果 `chunk_size` 小于或等于 0。
            ValueError: 如果 `chunk_overlap` 小于 0。
            ValueError: 如果 `chunk_overlap` 大于 `chunk_size`。
        """
        if chunk_size <= 0:
            msg = f"chunk_size must be > 0, got {chunk_size}"
            raise ValueError(msg)
        if chunk_overlap < 0:
            msg = f"chunk_overlap must be >= 0, got {chunk_overlap}"
            raise ValueError(msg)
        if chunk_overlap > chunk_size:
            msg = (
                f"Got a larger chunk overlap ({chunk_overlap}) than chunk size "
                f"({chunk_size}), should be smaller."
            )
            raise ValueError(msg)

        self._chunk_size = chunk_size
        self._chunk_overlap = chunk_overlap
        self._length_function = length_function
        self._keep_separator = keep_separator
        self._add_start_index = add_start_index
        self._strip_whitespace = strip_whitespace

    @abstractmethod
    def split_text(self, text: str) -> list[str]:
        # 此方法是抽象方法，具体的实现细节由子类来决定
        """将文本切分为多个组成部分。

        Args:
            text: 要切分的文本。

        Returns:
            文本块列表。
        """

    # 传入字符串列表，返回document对象列表
    def create_documents(
        self, 
        texts: list[str],
        metadatas: list[dict[Any, Any]] | None = None,
    ) -> list[Document]:
        """根据文本列表创建一组 `Document` 对象。

        Args:
            texts: 需要被切分并转换为文档的文本列表。
            metadatas: 可选的元数据列表，用于关联到每个文档。

        Returns:
            `Document` 对象列表。
        """
        metadatas_ = metadatas or [{}] * len(texts)
        documents = []
        for i, text in enumerate(texts):
            index = 0
            previous_chunk_len = 0
            for chunk in self.split_text(text):
                metadata = copy.deepcopy(metadatas_[i])
                if self._add_start_index:
                    offset = index + previous_chunk_len - self._chunk_overlap
                    index = text.find(chunk, max(0, offset))
                    metadata["start_index"] = index
                    previous_chunk_len = len(chunk)
                new_doc = Document(page_content=chunk, metadata=metadata)
                documents.append(new_doc)
        return documents

    def split_documents(self, documents: Iterable[Document]) -> list[Document]:
        """切分文档。

        Args:
            documents: 要切分的文档。

        Returns:
            切分后的文档列表。
        """
        texts, metadatas = [], []
        for doc in documents:
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)
        return self.create_documents(texts, metadatas=metadatas)

    def _join_docs(self, docs: list[str], separator: str) -> str | None:
        text = separator.join(docs)
        if self._strip_whitespace:
            text = text.strip()
        return text or None

    def _merge_splits(self, splits: Iterable[str], separator: str) -> list[str]:
        # 现在我们希望将这些较小的片段组合成中等大小的
        # 文本块，以便发送给 LLM。
        # 实现细节省略...

    @classmethod
    def from_huggingface_tokenizer(
        cls, tokenizer: PreTrainedTokenizerBase, **kwargs: Any
    ) -> TextSplitter:
        """使用 Hugging Face tokenizer 计算长度的文本切分器。

        Args:
            tokenizer: 要使用的 Hugging Face tokenizer。

        Returns:
            一个使用 Hugging Face tokenizer 进行长度计算的 `TextSplitter` 实例。
        """
        # 实现细节省略...

    @classmethod
    def from_tiktoken_encoder(
        cls,
        encoding_name: str = "gpt2",
        model_name: str | None = None,
        allowed_special: Literal["all"] | AbstractSet[str] = set(),
        disallowed_special: Literal["all"] | Collection[str] = "all",
        **kwargs: Any,
    ) -> Self:
        """使用 `tiktoken` 编码器计算长度的文本切分器。

        Args:
            encoding_name: 要使用的 tiktoken 编码名称。
            model_name: 要使用的模型名称。
                如果提供该参数，它将覆盖 `encoding_name`。
            allowed_special: 编码过程中允许的特殊 token。
            disallowed_special: 编码过程中不允许的特殊 token。

        Returns:
            一个使用 tiktoken 进行长度计算的 `TextSplitter` 实例。

        Raises:
            ImportError: 如果未安装 tiktoken 包。
        """
        # 实现细节省略...

    @override
    def transform_documents(
        self, documents: Sequence[Document], **kwargs: Any
    ) -> Sequence[Document]:
        """通过切分文档来转换文档序列。

        Args:
            documents: 要切分的文档序列。

        Returns:
            切分后的文档列表。
        """
        return self.split_documents(list(documents))

## 具体实现
### CharacterTextSplitter:Split by character
参数情况说明：
- chunk_size ：每个切块的最大字符数量，默认值为4000。
  
- chunk_overlap ：相邻两个切块之间的最大重叠字符数量，默认值为200。为了保证段之间语义完整，可以设置每个块之间有一部分重叠。
  
- separator ：分割使用的分隔符，默认值为"\n\n"。
  
- length_function ：用于计算切块长度的方法。默认赋值为父类TextSplitter的len函数。

In [ ]:
### 字符串文本的分割
### 若必须禁用分隔符（如处理无空格文本），需容忍实际块长略小于 chunk_size （尤其对中文）
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""
# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    #length_function=len,
    separator="" # 设置为空字符串时，表示禁用分隔符优先
)
# 4.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：49
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发
--------------------------------------------------
块 2:长度：23
象，使开发者
能够更容易地构建复杂的应用程序。
--------------------------------------------------
